In [18]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import laspy
import random

def process_las_files_split3(
    directory, out_root, chunk_size=100000, tile_name="tile",
    train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios müssen zusammen 1.0 ergeben"

    files = [os.path.join(directory, f) for f in os.listdir(directory)
             if f.endswith(".las") or f.endswith(".laz")]
    header = []
    chunk_data_buffer = []
    chunk_counter = 0
    random.seed(seed)

    for file_idx, file in enumerate(tqdm(files, desc="Lade .las-Dateien")):
        try:
            las = laspy.read(file)
            dims = list(las.points.point_format.dimension_names)
            scale = las.header.scale
            offset = las.header.offset
            if not header:
                header = dims
            total_points = len(las.points)
        except Exception as e:
            print(f"\nFehler beim Lesen von {file}: {e}")
            continue

        start = 0
        while start < total_points:
            end = min(start + chunk_size, total_points)
            chunk_data = []
            valid_chunk = True
            for d in dims:
                try:
                    data_array = np.array(las.points[d][start:end], dtype=float)
                    if d in ["X", "Y", "Z"]:
                        idx = ["X", "Y", "Z"].index(d)
                        data_array = data_array * scale[idx] + offset[idx]
                    if data_array.size != (end - start):
                        print(f"\nWarnung: Inkonsistente Punktzahl in Chunk {chunk_counter}")
                        valid_chunk = False
                        break
                    chunk_data.append(data_array)
                except Exception as e:
                    print(f"\nFehler bei Dimension '{d}': {e}")
                    valid_chunk = False
                    break

            if not valid_chunk or len(chunk_data) != len(dims):
                start = end
                continue

            chunk = np.vstack(chunk_data).T
            chunk_df = pd.DataFrame(chunk, columns=header)

            feature_cols = ["X", "Y", "Z", "intensity", "return_number", "number_of_returns"]
            label_col = "classification"

            if not all(col in chunk_df.columns for col in feature_cols + [label_col]):
                print(f"\n❌ Chunk {chunk_counter} übersprungen – fehlende Spalten.")
                start = end
                continue

            features = chunk_df[feature_cols].to_numpy()
            labels = chunk_df[label_col].to_numpy().astype(np.int32)

            chunk_data_buffer.append((features, labels, f"{tile_name}_chunk{chunk_counter}"))
            chunk_counter += 1

            start = end

    if not chunk_data_buffer:
        print("\n⚠️ Keine gültigen Chunks extrahiert.")
        return

    # Shuffle and split
    random.shuffle(chunk_data_buffer)
    n_total = len(chunk_data_buffer)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    n_test = n_total - n_train - n_val

    splits = {
        "train": chunk_data_buffer[:n_train],
        "val": chunk_data_buffer[n_train:n_train + n_val],
        "test": chunk_data_buffer[n_train + n_val:]
    }

    for split_name, chunks in splits.items():
        split_dir = os.path.join(out_root, split_name)
        os.makedirs(split_dir, exist_ok=True)
        for features, labels, name in chunks:
            points_path = os.path.join(split_dir, f"{name}_points.npy")
            labels_path = os.path.join(split_dir, f"{name}_labels.npy")
            np.save(points_path, features)
            np.save(labels_path, labels)

    print(f"\n✅ Aufteilung abgeschlossen:")
    print(f"  🔹 Train: {n_train} Chunks")
    print(f"  🔸 Val:   {n_val} Chunks")
    print(f"  🔻 Test:  {n_test} Chunks")
    print(f"  📂 Gespeichert in: '{out_root}/{{train,val,test}}'")

In [19]:
process_las_files_split3(
    directory="C:/Users/Nicola/Desktop/zurich_laz",
    out_root="./processed_chunks",
    chunk_size=100_000,
    tile_name="tileCH",
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
)



Lade .las-Dateien: 100%|██████████| 25/25 [01:24<00:00,  3.36s/it]



✅ Aufteilung abgeschlossen:
  🔹 Train: 1290 Chunks
  🔸 Val:   276 Chunks
  🔻 Test:  278 Chunks
  📂 Gespeichert in: './processed_chunks/{train,val,test}'


In [20]:
import tensorflow as tf
import numpy as np
import os
import glob

def load_dataset_from_separate_npy(
    directory,
    batch_size=8,
    shuffle=True,
    shuffle_buffer=100,
    prefetch=True
):
    """
    Lädt getrennte .npy-Dateien (points + labels) aus einem Verzeichnis
    und gibt ein tf.data.Dataset zurück.

    Arguments:
        directory:        Ordner mit *_points.npy und *_labels.npy
        batch_size:       Anzahl Chunks pro Batch (nicht Punkte!)
        shuffle:          Ob die Reihenfolge der Chunks zufällig sein soll
        shuffle_buffer:   Anzahl Chunks zum Shuffeln (nur bei shuffle=True)
        prefetch:         Ob `prefetch()` verwendet wird

    Returns:
        tf.data.Dataset mit batches von:
            points: (batch_size, N, num_features)
            labels: (batch_size, N)
    """
    point_files = sorted(glob.glob(os.path.join(directory, "*_points.npy")))
    label_files = sorted(glob.glob(os.path.join(directory, "*_labels.npy")))

    if len(point_files) != len(label_files):
        raise ValueError("Nicht gleich viele *_points.npy und *_labels.npy Dateien!")

    file_pairs = list(zip(point_files, label_files))

    def load_point_label_pair(point_path, label_path):
        def _load_numpy(p_path, l_path):
            points = np.load(p_path.decode("utf-8")).astype(np.float32)
            labels = np.load(l_path.decode("utf-8")).astype(np.int32)
            return points, labels
        points, labels = tf.numpy_function(
            _load_numpy, [point_path, label_path], [tf.float32, tf.int32]
        )
        points.set_shape([None, 6])  # z. B. 1024 Punkte mit 6 Features
        labels.set_shape([None])     # z. B. 1024 Klassenlabels
        return points, labels

    dataset = tf.data.Dataset.from_tensor_slices(file_pairs)
    if shuffle:
        dataset = dataset.shuffle(buffer_size=shuffle_buffer)
    dataset = dataset.map(load_point_label_pair, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    if prefetch:
        dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


In [21]:
train_dataset = load_dataset_from_npy_chunks(
    directory="./processed_chunks/train",
    batch_size=8
)

val_dataset = load_dataset_from_npy_chunks(
    directory="./processed_chunks/val",
    batch_size=8,
    shuffle=False
)

In [22]:
import os
import numpy as np
import tensorflow as tf
from pointnet2.models import pointnet2_sem_seg
from pointnet2.utils import provider


# === KONFIGURATION ===
NUM_CLASSES = 9  # z. B. Wasser, Boden, Vegetation, Gebäude, etc.
NUM_POINT = 4096
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 0.001

TRAIN_DIR = '/path/to/npy/train'
VAL_DIR = '/path/to/npy/val'


# === DATENLADEN ===
def load_npy_dataset(directory):
    files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".npy")]
    def generator():
        for file in files:
            data = np.load(file, allow_pickle=True).item()
            yield data['points'], data['labels']

    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(NUM_POINT, None), dtype=tf.float32),
            tf.TensorSpec(shape=(NUM_POINT,), dtype=tf.int32)
        )
    )


def prepare_dataset(ds):
    return (ds
        .shuffle(100)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE))

train_ds = prepare_dataset(load_npy_dataset(TRAIN_DIR))
val_ds = prepare_dataset(load_npy_dataset(VAL_DIR))


# === MODELL ===
model = pointnet2_sem_seg.get_model(NUM_CLASSES)
loss_fn = pointnet2_sem_seg.get_loss()

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)


# === TRAININGSLOOP ===
@tf.function
def train_step(points, labels):
    with tf.GradientTape() as tape:
        logits = model(points, training=True)
        loss = loss_fn(logits, labels)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

@tf.function
def val_step(points, labels):
    logits = model(points, training=False)
    loss = loss_fn(logits, labels)
    return loss

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    train_loss = []
    for points, labels in train_ds:
        loss = train_step(points, labels)
        train_loss.append(loss.numpy())

    val_loss = []
    for points, labels in val_ds:
        loss = val_step(points, labels)
        val_loss.append(loss.numpy())

    print(f"Train loss: {np.mean(train_loss):.4f} | Val loss: {np.mean(val_loss):.4f}")


ModuleNotFoundError: No module named 'pointnet2'